In [89]:
from EKFSlam_KL import EKF_SLAM
import matplotlib.pyplot as plt
import numpy as np


In [90]:
def angle_dist(b, a):
	theta = b - a
	while theta < -np.pi:
		theta += 2. * np.pi
	while theta > np.pi:
		theta -= 2. * np.pi
	return theta


In [91]:
# Motion model and derivative
def motion_model(x, u, dt = 0.1):
    out = np.copy(x)
    out[:3, :] += u # only the robot state is affected
    return out

def motion_model_deriv(x, u, dt = 0.1):
    return np.eye(x.shape[0])

# Helper function: get action u from actual odometry poses
def u_from_poses(xn, xp, dt = 0.1):
    dx = xn[1, 0] - xp[1, 0]
    dy = xn[2, 0] - xp[2, 0]
    dtheta = angle_dist(xn[0, 0], xp[0, 0])
    return np.array([[dtheta, dx, dy]]).T

# Odometry integration
def integrate(p, Vb, dt):
    # p: previous/current odometry estimate: p = [θ, x, y]^T
    # Vb: commanded velocity (given the differential drive kinematics) in body frame
    # dt: timestep to perform Euler integration
    ### TO-DO: Compute and return the next odometry pose
    ### ANSWER: Insert code here
    new_p = np.copy(p)
    new_p[0, 0] += Vb[0, 0] * dt
       
    new_p[1, 0] +=  Vb[1, 0]* dt  * np.cos(p[0, 0]) - Vb[2, 0] * dt * np.sin(p[0, 0])
    new_p[2, 0] += Vb[1, 0]* dt  * np.sin(p[0, 0]) + Vb[2, 0] * dt * np.cos(p[0, 0])
    
    return new_p
    ### END of ANSWER

In [92]:
# Observation model and derivative
def observation_model(x, landmark_pose):
    # x: robot pose (a 3x1 numpy array)
    # landmark_pose (a 2x1 numpy array)
    ### TO-DO: Compute the observation model! Return a 2x1 numpy array with [[distance], [relative angle]]
    ### ANSWER: Insert code here
    mx = landmark_pose[0]
    my = landmark_pose[1]
    theta = x[0,0]
    rx = x[1,0]
    ry = x[2,0]

    dist = np.sqrt(( mx - rx)**2 +(my-ry)**2)

    phi = np.atan2(my - ry, mx - rx) - theta
    phi = angle_dist(phi, 0) 

    return np.array([dist,phi])
    ### END of ANSWER

def observation_model_deriv(x, landmark_pose):
    ### TO-DO: Compute the derivative of the above wrt the robot pose (x), and landmark pose
    ### Store the result in the `jac` variable
    jac = np.zeros((2, 5))
    ### ANSWER: Insert code here
    # Extract robot and landmark poses
    mx = landmark_pose[0, 0]
    my = landmark_pose[1, 0]
    theta = x[0, 0]
    rx = x[1, 0]
    ry = x[2, 0]

    dx = mx - rx
    dy = my - ry
    dist = np.sqrt(dx**2 + dy**2)
    dist_squared = dx**2 + dy**2

    jac[0, 0] = 0  
    jac[0, 1] = -dx / dist  
    jac[0, 2] = -dy / dist  
    jac[0, 3] = dx / dist   
    jac[0, 4] = dy / dist   

    jac[1, 0] = -1  
    jac[1, 1] = dy / dist_squared  
    jac[1, 2] = -dx / dist_squared  
    jac[1, 3] = -dy / dist_squared  
    jac[1, 4] = dx / dist_squared  
    
    
    ### END of ANSWER

    return jac

In [93]:
def robot_landmark_2d_measurement(robot_state, landmark_state):
	# robot_state: x = [θ, x, y]^T (numpy 3x1 array)
	# landmark_state = (m_x, m_y) (tuple0
	### TO-DO: Compute the 2D measurement for the robot_state and the landmark_state
	### You should return a tuple with (distance, relative_angle)
	### ANSWER: Insert code here
	mx = landmark_state[0]
	my = landmark_state[1]
	theta = robot_state[0]

	rx = robot_state[1]
	ry = robot_state[2]
	dist = np.sqrt(( mx - rx)**2 +(my-ry)**2)

	phi = np.atan2(my - ry, mx - rx) - theta
	phi = angle_dist(phi, 0) 

	return (dist, phi)

In [94]:
def robot_move(u, state, dt):

	v, w = u
	x, y, theta = state
	# Update state using odometry model
	x += v * np.cos(theta) * dt
	y += v * np.sin(theta) * dt
	theta += w * dt
	R = np.diag([0.1, 0.1, np.deg2rad(0.1)]) ** 2
	Q = np.diag([1, np.deg2rad(5)]) ** 2
	# Add noise to the state
	x += np.random.normal(0, np.sqrt(R[0,0]))
	y += np.random.normal(0, np.sqrt(R[1,1]))
	theta += np.random.normal(0, np.sqrt(R[2,2]))
	result = np.array([[x], [y], [theta]])
	print("robot_move: ", result)
	return result

In [95]:
def camera(x, landmarks, width = 2. * np.pi, noise = 1e-3):
    detects = []

    ### TO-DO: Compute all possible detects. You should fill the list detects with tuples of the form (distance, relative_angle, landmark id)
    ### The landmark ids should be 1-based.
    ### You should add Gaussian noise to the distance and relative angle with zero mean and `noise` as variance
    for k in range(len(landmarks)):
        ### ANSWER: Insert code here
        dist, phi = robot_landmark_2d_measurement(x,landmarks[k])
        if phi < width/2 and phi > - width / 2:
            # i have detected a landmark
            dist += np.random.normal(0,noise)
            phi += np.random.normal(0,noise)
            detects.append((dist,phi,k+1))
        ### END of ANSWER
    return detects

In [96]:
# Initialize the robot
slam = EKF_SLAM()

# Set parameters if needed
slam.set_time_step(0.2)  # 100ms time step
time_steps = 10  # Number of time steps to simulate
# Example control and measurement
u = [0, np.deg2rad(10)]  # Move forward at 1m/s with 5deg/s rotation
landmarks = [(4., 4.), (4., 0.), (4., -4.), (0., -4.), (-4., -4.), (-4., 0.), (-4., 4.), (0., 4.)]
angle_of_view = np.pi / 4  # 45 degrees
# Initialize the robot pose
robot_pose = np.array([[0.], [0.], [0.]])  # Initial robot pose
real_trajectory = np.zeros((3,time_steps))
print("real_trajectory: ", real_trajectory)
estimated_trajectory = np.zeros((3, time_steps))
real_landmarks = np.zeros((2, len(landmarks)))
estimated_landmarks = np.zeros((2, len(landmarks)))

for timestep in range(time_steps):
	# Simulate robot motion
	# print(f"robot pose: {robot_pose}, landmarks: {landmarks}, input: {u}")
	robot_pose = robot_move(u, robot_pose, slam.dt)
	print("robot pose after move: ", robot_pose[0])
	real_trajectory[:,timestep] = robot_pose.flatten()
	# read the camera
	detects = camera(robot_pose, landmarks, angle_of_view)
	# print("detected landmarks: ", detects)
	
		



real_trajectory:  [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
robot_move:  [[[-0.01212667]]

 [[-0.16107267]]

 [[ 0.03277375]]]
robot pose after move:  [[-0.01212667]]
robot_move:  [[[[ 0.14876311]]]


 [[[-0.13592793]]]


 [[[ 0.06804696]]]]
robot pose after move:  [[[0.14876311]]]
robot_move:  [[[[[ 0.19150877]]]]



 [[[[-0.25418932]]]]



 [[[[ 0.10678706]]]]]
robot pose after move:  [[[[0.19150877]]]]
robot_move:  [[[[[[ 0.15885882]]]]]




 [[[[[-0.45114455]]]]]




 [[[[[ 0.14242208]]]]]]
robot pose after move:  [[[[[0.15885882]]]]]
robot_move:  [[[[[[[ 0.2127682 ]]]]]]





 [[[[[[-0.52528922]]]]]]





 [[[[[[ 0.17934006]]]]]]]
robot pose after move:  [[[[[[0.2127682]]]]]]
robot_move:  [[[[[[[[ 0.37903324]]]]]]]






 [[[[[[[-0.48521558]]]]]]]






 [[[[[[[ 0.2159889 ]]]]]]]]
robot pose after move:  [[[[[[[0.37903324]]]]]]]
robot_move:  [[[[[[[[[ 0.22086535]]]]]]]]







 [[[[[[[[-0.39339039]]]]]]]]







 [[[[[[[[ 0